# AGender-MTL - Step 1: the benchmark (CPU)

- **Data:** the 29 raw LAT attributes; log(1+x) and standardisation for skewed numbers, one-hot for columns
  with at most 10 values, frequency encoding for the others, missing values as their own category; all fitted
  on the training rows only.
- **Split:** 70 / 15 / 15 stratified by the six age groups, seeds 42, 7, 123, 2024, 99.
- **QUBO:** NMI relevance to the joint gender x age label, alpha 1, beta 0.05, lambda 1.5, k 25; simulated
  annealing, 6 restarts x 20,000 flips, T 5 -> 0.01.
- **Models:** the multi-task network (NumPy), Logistic Regression, Random Forest, Gradient Boosting
  (histogram version for 48 classes).
- **Age schemes:** 3, 6, 10 and 48 classes; QUBO sensitivity sweep (validation, 6 classes).

**Kaggle settings:** Accelerator = **None** (CPU). Internet = Off is fine.
**Inputs to add:** `agender-mtl-code` and `agender-mtl-step0`.
**Expected run time:** about 30-45 minutes. Every scheme is written as soon as it is done.

In [ ]:
import os
import sys
from pathlib import Path

# The code is attached as the Kaggle dataset `agender-mtl-code`. Find the `agender_mtl` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('benchmark.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-mtl-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-mtl-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender_mtl' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender_mtl`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender_mtl').exists():
        os.symlink(package.parent, code_root / 'agender_mtl')

sys.path.insert(0, str(code_root))
import agender_mtl
print('Code loaded from', Path(agender_mtl.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_mtl_outputs/step1')
TIME_BUDGET_MINUTES = 150
SCHEMES = ['6', '3', '10', '48']      # 6 first: it holds the main table
RUN_SWEEP = True

In [ ]:
import json

from agender_mtl import environment, step1
from agender_mtl.runner import StepRunner

runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}

# Kaggle unpacks zip files when an output is saved as a dataset, so look for the zip or the csv inside it.
lat_path = find_file('lat_s1.zip', INPUT) or find_file('age-gender-pred-dataset.csv', INPUT)
print('LAT file:', lat_path)
if lat_path is None:
    raise RuntimeError('Attach the saved output of Step 0 (agender-mtl-step0): it holds the LAT file.')

summary['environment'] = runner.run('measure the machine', environment.collect)
raw = runner.run('read LAT', step1.raw_lat, lat_path)
print(raw.shape)

In [ ]:
for scheme in SCHEMES:
    result = runner.run(f'{scheme}-class scheme, five seeds', step1.run_scheme_all_seeds, raw, scheme, OUTPUT)
    summary[f'scheme_{scheme}'] = result
    if result:
        print('MTL best epochs:', result['mtl_best_epochs'])
        for name, scores in result['test_means'].items():
            print(f'  {name:28s}', scores)

In [ ]:
if RUN_SWEEP:
    summary['sweep'] = runner.run('QUBO sweep (validation, 6 classes)', step1.run_sweep_all_seeds, raw, OUTPUT)
    for row in summary['sweep'] or []:
        print(row)

In [ ]:
# ---- Summary (always written) ----
summary['steps'] = runner.steps
with open(OUTPUT / 'step1_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print('Failed or skipped steps:', runner.failed() or 'none')